# Mi8 Multipath — Classification Model

Trains and evaluates classifiers to detect multipath from Mi8 GNSS measurements. The label (`MultipathIndicator`) comes directly from the device hardware — no ground-truth trajectory comparison is needed.

**Input:** `data/03_processed/mi8_training_features.csv`

## 1. Library Import & Data Loading

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc, classification_report
)

BASE_DIR  = os.path.abspath(os.path.join(os.getcwd(), '../..'))
DATA_PATH = os.path.join(BASE_DIR, 'data/03_processed/mi8_training_features.csv')

df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df):,} rows  |  shape: {df.shape}')
print('Class distribution:', df['MultipathIndicator'].value_counts().to_dict())
df.head()

Loaded 644,560 rows  |  shape: (644560, 18)
Class distribution: {0: 322280, 1: 322280}


,ReceivedSvTimeUncertaintyNanos,Cn0DbHz,PseudorangeRateMetersPerSecond,PseudorangeRateUncertaintyMetersPerSecond,AccumulatedDeltaRangeState,AccumulatedDeltaRangeMeters,AccumulatedDeltaRangeUncertaintyMeters,AgcDb,Const_BeiDou,Const_GLONASS,Const_GPS,Const_Galileo,Const_QZSS,Cn0_Cat_VeryWeak,Cn0_Cat_Weak,Cn0_Cat_Medium,Cn0_Cat_Strong,MultipathIndicator
0,2.134980,-2.040796,-0.057844,-0.159274,-0.426389,-1.413331,1.642554,0.205533,False,False,True,False,False,False,True,False,False,0
1,2.569085,-2.314867,0.021749,-0.157830,-0.426389,0.246669,1.642554,0.205533,False,False,True,False,False,True,False,False,False,1
2,3.165979,-2.606252,-0.000442,-0.156263,-0.426389,-0.244660,1.642554,0.205533,False,False,True,False,False,True,False,False,False,1
3,0.724140,-0.871047,-0.013401,-0.179386,-0.814269,-0.230825,-0.608808,0.205533,False,False,True,False,False,False,True,False,False,0
4,2.352033,-2.169863,0.003072,-0.158565,-0.426389,0.128192,1.642554,0.205533,False,False,True,False,False,True,False,False,False,0


## 2. Train / Test Split

Stratified 80/20 split to maintain the 50/50 SMOTE-balanced class ratio.

In [ ]:
X = df.drop(columns=['MultipathIndicator'])
y = df['MultipathIndicator'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Train class balance: {y_train.value_counts(normalize=True).round(3).to_dict()}')

## 2b. Preprocessing (leak-free)\n\nAll data-dependent transforms (imputation, scaling, SMOTE) are fitted **on training data only**, then applied to the test set — never the reverse.

In [ ]:
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# 1. Impute AgcDb using the TRAINING median only
if 'AgcDb' in X_train.columns:
    agc_median = X_train['AgcDb'].median()
    X_train['AgcDb'] = X_train['AgcDb'].fillna(agc_median)
    X_test['AgcDb']  = X_test['AgcDb'].fillna(agc_median)
    print(f'AgcDb imputed with train median: {agc_median:.4f}')

# 2. Scale continuous features — fit on train only, transform both
bool_cols = X_train.select_dtypes(include='bool').columns.tolist()
cont_cols  = [c for c in X_train.select_dtypes(include=np.number).columns
              if c not in bool_cols]

scaler = StandardScaler()
X_train[cont_cols] = scaler.fit_transform(X_train[cont_cols])
X_test[cont_cols]  = scaler.transform(X_test[cont_cols])        # transform only
print(f'Scaled {len(cont_cols)} features on train, applied to test.')

# 3. SMOTE on training data only
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)
print(f'After SMOTE — train shape: {X_train.shape}  |  balance: {pd.Series(y_train).value_counts().to_dict()}')

## 3. Baseline Models

Initialises a Random Forest, Gradient Boosting, and SVM before any hyperparameter tuning.

In [ ]:
# SVC is O(n²–n³) — completely impractical on 600k+ rows.
# SGDClassifier (hinge loss) is a linear SVM that scales O(n) and finishes in seconds.
# HistGradientBoostingClassifier is sklearn's fast histogram-based GB (like LightGBM).
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV

rf_clf  = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
hgb_clf = HistGradientBoostingClassifier(max_iter=200, random_state=42)

# Wrap SGD in CalibratedClassifierCV to get predict_proba for ROC curves
sgd_base = SGDClassifier(loss='hinge', max_iter=1000, random_state=42, n_jobs=-1)
sgd_clf  = CalibratedClassifierCV(sgd_base, cv=3)

print('Models initialised:')
print(' RF: ', rf_clf)
print(' HGB:', hgb_clf)
print(' SGD:', sgd_clf)

## 4. Hyperparameter Tuning — Random Forest

`GridSearchCV` with 5-fold stratified cross-validation, optimising for F1 score.

In [ ]:
TUNE_SAMPLE = 20_000
tune_idx = y_train.sample(n=min(TUNE_SAMPLE, len(y_train)), random_state=42).index
X_tune = X_train.loc[tune_idx]
y_tune = y_train.loc[tune_idx]
print(f'Tuning subsample: {len(X_tune):,} rows  |  balance: {y_tune.value_counts(normalize=True).round(3).to_dict()}')

param_grid_rf = {
    'n_estimators':     [100, 200],
    'max_depth':        [10, None],
    'min_samples_leaf': [1, 2],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid_rf, cv=cv, scoring='f1', n_jobs=-1, verbose=1
)
grid_rf.fit(X_tune, y_tune)

best_rf = RandomForestClassifier(**grid_rf.best_params_, random_state=42, n_jobs=-1)
best_rf.fit(X_train, y_train)

print('Best RF params:', grid_rf.best_params_)
print('CV F1 on subsample:', round(grid_rf.best_score_, 4))

## 5. Train All Models & Generate Predictions

In [ ]:
hgb_clf.fit(X_train, y_train)
sgd_clf.fit(X_train, y_train)

models = {
    'Random Forest (tuned)':      best_rf,
    'Hist Gradient Boosting':     hgb_clf,
    'SGD Linear SVM':             sgd_clf,
}

preds = {name: m.predict(X_test)            for name, m in models.items()}
probs = {name: m.predict_proba(X_test)[:, 1] for name, m in models.items()}

print('All models trained and predictions generated.')

## 6. Evaluation Metrics

In [ ]:
def print_metrics(y_true, y_pred, name):
    print(f'\n--- {name} ---')
    print(classification_report(y_true, y_pred, target_names=['Clean', 'Multipath']))
    return f1_score(y_true, y_pred)

f1_scores = {}
for name, pred in preds.items():
    f1_scores[name] = print_metrics(y_test, pred, name)

## 7. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, pred) in zip(axes, preds.items()):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Clean', 'Multipath'],
                yticklabels=['Clean', 'Multipath'])
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.suptitle('Confusion Matrices', fontsize=13)
plt.tight_layout()
plt.show()

## 8. ROC Curves

In [ ]:
plt.figure(figsize=(9, 7))

for name, prob in probs.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name}  (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Mi8 Multipath Classifier')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()

## 9. F1 Score Comparison

In [ ]:
plt.figure(figsize=(8, 4))
bars = plt.barh(list(f1_scores.keys()), list(f1_scores.values()), color='steelblue')
for bar, val in zip(bars, f1_scores.values()):
    plt.text(val + 0.002, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center')
plt.xlabel('F1 Score')
plt.title('F1 Score Comparison')
plt.xlim(0, 1.05)
plt.tight_layout()
plt.show()

best_name = max(f1_scores, key=f1_scores.get)
print(f'Best model: {best_name}  (F1 = {f1_scores[best_name]:.4f})')

## 10. Feature Importance (Random Forest)

Shows which GNSS measurement features contribute most to the multipath detection decision.

In [ ]:
importances = best_rf.feature_importances_
imp_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
imp_df.sort_values('Importance', ascending=True, inplace=True)

plt.figure(figsize=(10, max(4, len(imp_df) * 0.35)))
plt.barh(imp_df['Feature'], imp_df['Importance'], color='steelblue')
plt.xlabel('Importance Score')
plt.title('Feature Importances — Tuned Random Forest')
plt.tight_layout()
plt.show()

print(imp_df.sort_values('Importance', ascending=False).to_string(index=False))

## 11. Cross-Validation Summary

Final sanity check with 5-fold stratified CV on the best model to confirm the test-set result generalises.

In [ ]:
from sklearn.model_selection import cross_val_score

CV_SAMPLE = 50_000
# Sample without stratify — data is already 50/50 from SMOTE so balance holds.
# Use .loc not .iloc: sample().index returns label indices, not positional integers.
cv_idx = y.sample(n=CV_SAMPLE, random_state=42).index
X_cv = X.loc[cv_idx]
y_cv = y.loc[cv_idx]

cv_scores = cross_val_score(
    best_rf, X_cv, y_cv,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1', n_jobs=-1
)

print(f'5-fold CV F1 (50k subsample): {cv_scores.round(4)}')
print(f'Mean: {cv_scores.mean():.4f}  ±  {cv_scores.std():.4f}')